# 01 · Data pipeline: extract, clean, join
**Owner:** Yaye Song

Walks through `src/data_prep.py` one step at a time. The script does the same thing in one go:
`python src/data_prep.py`. Cleaning changes go in the script, then re-run this notebook.

**Still to do (from the proposal):**
- [ ] Treat stats not recorded in early eras (SF, HBP, CS before ~1954) as missing, not zero
- [ ] Flag seasons lost to WWII service (e.g. Ted Williams, Hank Greenberg)
- [ ] Edge cases: late comebacks pulling mostly pre-war players in (e.g. Dizzy Dean, 1947); sensitivity check with "most of career after 1946"
- [ ] Document every exclusion with counts: careers ending before 1946, Negro Leagues-only players, careers ending after 2010

In [1]:
import sys
sys.path.append('../src')
import pandas as pd
import data_prep as dp

## Step 1: Extract

**Source:** [Lahman Baseball Database](https://sabr.org/lahman-database), 2025 release, published by SABR. It has 28 linked CSV tables covering 1871–2025, licensed CC BY-SA 3.0. The repo stores it unmodified as `data/raw/lahman_2025.zip` (11 MB, 41 MB unzipped), including the data dictionary `readme2025.txt`.

`ensure_raw_data()` unzips it into `data/raw/lahman_2025/` the first time (git ignores that folder). To open it by hand instead:

| Where | How |
|---|---|
| Windows | Right-click `lahman_2025.zip` → **Extract All…** → extract to `data/raw/` |
| macOS | Double-click it and keep the `lahman_2025` folder in `data/raw/` |
| Linux / WSL | `unzip data/raw/lahman_2025.zip -d data/raw/` |
| pandas, without unzipping | `pd.read_csv(zipfile.ZipFile(path).open('lahman_2025/People.csv'))`. A plain `pd.read_csv` on the zip fails because it holds many files. |

To check the files are intact: `cd data/raw/lahman_2025 && sha256sum -c ../SHA256SUMS.txt`

In [2]:
raw = dp.ensure_raw_data()
files = sorted(raw.glob('*.csv'))
print(f'{len(files)} CSV files in {raw}')
pd.DataFrame({'file': [f.name for f in files], 'MB': [round(f.stat().st_size / 1e6, 2) for f in files]})

27 CSV files in /home/alex/data_science_class/group/data/raw/lahman_2025


,file,MB
0,AllstarFull.csv,0.26
1,Appearances.csv,7.83
2,AwardsManagers.csv,0.01
3,AwardsPlayers.csv,0.59
4,AwardsShareManagers.csv,0.03
5,AwardsSharePlayers.csv,0.36
6,Batting.csv,8.00
7,BattingPost.csv,1.16
8,CollegePlaying.csv,0.43
9,Fielding.csv,8.77


In [3]:
# Tables this project uses
pd.DataFrame({t: dp.load(t).shape for t in ['HallOfFame', 'Batting', 'Pitching', 'People',
                                             'AwardsPlayers', 'AllstarFull', 'Appearances']},
             index=['rows', 'columns']).T

,rows,columns
HallOfFame,6426,9
Batting,128598,22
Pitching,57630,30
People,24270,25
AwardsPlayers,12667,6
AllstarFull,6425,8
Appearances,128512,21


## Step 2: Clean

Each raw table becomes one row per player:
- **Leagues:** only AL/NL seasons are kept. Negro Leagues, National Association and Federal League statistics aren't comparable.
- **Mid-season trades:** a player's rows for one season (one per team) are summed into career totals.
- **Career span:** number of seasons, first and final AL/NL season.
- **Honours:** counts of MVP, Cy Young, Gold Glove, Rookie of the Year, World Series MVP and All-Star seasons.
- **Position:** the position with the most career games.
- **Label:** Hall of Fame rows with `category == "Player"` only (managers, executives and umpires are dropped), plus the year and route of first induction.

In [4]:
tables = dp.clean_all()
pd.DataFrame({'rows (players)': {k: len(v) for k, v in tables.items()},
              'columns': {k: v.shape[1] if v.ndim > 1 else 1 for k, v in tables.items()}})

,rows (players),columns
span,20423,3
batting,20423,10
pitching,10599,8
honours,2456,6
position,24011,1
hof,281,2
people,24270,2


In [5]:
tables['batting'].head()

,G,PA,R,H,HR,RBI,SB,AVG,OBP,SLG
playerID,,,,,,,,,,
aardsda01,331,4.0,0,0,0,0.0,0.0,0.000000,0.000000,0.000000
aaronha01,3298,13919.0,2174,3771,755,2297.0,240.0,0.304998,0.373949,0.554513
aaronto01,437,1036.0,102,216,13,94.0,9.0,0.228814,0.291506,0.327331
aasedo01,448,5.0,0,0,0,0.0,0.0,0.000000,0.000000,0.000000
abadan01,15,25.0,1,2,0,0.0,0.0,0.095238,0.240000,0.095238


In [6]:
tables['hof']['hof_route'].value_counts()

hof_route
BBWAA                             137
Veterans                           77
Old Timers                         25
Negro League                       21
Veterans - Golden Age Era           4
Run Off                             3
Veterans - Modern Baseball Era      3
Classic Baseball Era                2
Veterans - Today's Game Era         2
Special Election                    2
Veterans - Pre-1943                 1
Contemporary Baseball Era           1
Contemporary Era                    1
Veterans - Golden Era               1
Veterans - Pre-Integration Era      1
Name: count, dtype: int64

## Step 3: Join

Everything is left-joined onto the career span by `playerID`, then:
- **Eligibility:** at least 10 AL/NL seasons (the Hall of Fame minimum) and a final season in 1946–2010, so everyone has had a full ballot window. The whole career counts, including seasons before 1946.
- **Missing values:** a counting stat with no row means none, so it becomes 0. Rate stats stay NaN where they don't apply (batting rates for pitchers, ERA and WHIP for hitters). Imputation is a modelling decision.
- **Split:** `train` = final season 1946–1995, `test` = 1996–2010.

In [7]:
df = dp.join_all(tables)
print(f'All players with an AL/NL season: {len(tables["span"]):,} -> eligible: {len(df):,}')
df.head()

All players with an AL/NL season: 20,423 -> eligible: 2,110


,playerID,name,first_year,final_year,pos,split,seasons,G,PA,R,...,pos_dh,AllStar,MVP,CyYoung,GoldGlove,ROY,WS_MVP,HOF,hof_year,hof_route
0,aaronha01,Hank Aaron,1954,1976,of,train,23,3298,13919,2174,...,0,21,1,0,3,0,0,1,1982,BBWAA
1,aasedo01,Don Aase,1977,1990,p,train,13,448,5,0,...,0,1,0,0,0,0,0,0,<NA>,NaN
2,abbotgl01,Glenn Abbott,1973,1984,p,train,11,248,0,0,...,0,0,0,0,0,0,0,0,<NA>,NaN
3,abbotji01,Jim Abbott,1989,1999,p,test,10,263,21,0,...,0,0,0,0,0,0,0,0,<NA>,NaN
4,abbotpa01,Paul Abbott,1990,2004,p,test,11,162,20,2,...,0,0,0,0,0,0,0,0,<NA>,NaN


In [8]:
df.groupby('split').agg(players=('HOF', 'size'), inducted=('HOF', 'sum'), rate=('HOF', 'mean'))

,players,inducted,rate
split,,,
test,695,33,0.047482
train,1415,98,0.069258


### Missing values
Batting rates are NaN for pitchers, ERA/WHIP for position players. Imputation is left to modelling.

In [9]:
df.isna().sum()[lambda s: s > 0]

AVG           840
OBP           840
SLG           840
p_ERA        1270
p_WHIP       1270
hof_year     1979
hof_route    1979
dtype: int64

### Columns of the output table

| Group | Columns |
|---|---|
| Identifiers | `playerID`, `name`, `first_year`, `final_year`, `pos`, `split` |
| Performance | `seasons`, career batting and pitching stats, `is_pitcher`, `pos_*` |
| Honours | `AllStar`, `MVP`, `CyYoung`, `GoldGlove`, `ROY`, `WS_MVP` |
| Label | `HOF` (1 = inducted as a player, any route), `hof_year`, `hof_route` |

`hof_year` and `hof_route` describe the outcome. **Never use them as features.**

In [10]:
print('Performance:', dp.PERFORMANCE_FEATURES)
print('Honours:    ', dp.HONOURS_FEATURES)
print('Label:      ', dp.LABEL_COLUMNS, '(hof_year / hof_route are outcomes, never features)')

Performance: ['seasons', 'G', 'PA', 'R', 'H', 'HR', 'RBI', 'SB', 'AVG', 'OBP', 'SLG', 'p_W', 'p_L', 'p_SV', 'p_IPouts', 'p_SO', 'p_GS', 'p_ERA', 'p_WHIP', 'is_pitcher', 'pos_p', 'pos_c', 'pos_1b', 'pos_2b', 'pos_3b', 'pos_ss', 'pos_of', 'pos_dh']
Honours:     ['AllStar', 'MVP', 'CyYoung', 'GoldGlove', 'ROY', 'WS_MVP']
Label:       ['HOF', 'hof_year', 'hof_route'] (hof_year / hof_route are outcomes, never features)


## Save

Writes `data/processed/hof_players.csv` (ignored by git; anyone can rebuild it). Other notebooks load it with:

```python
import sys
sys.path.append('../src')
from data_prep import load_table, PERFORMANCE_FEATURES, HONOURS_FEATURES
df = load_table()
```

In [11]:
dp.PROCESSED.mkdir(parents=True, exist_ok=True)
df.to_csv(dp.OUTPUT, index=False)
print(f'Saved {dp.OUTPUT} ({df.shape[0]} x {df.shape[1]})')

Saved /home/alex/data_science_class/group/data/processed/hof_players.csv (2110 x 43)
